# 1. Problem Framing

## Objective

This notebook implements a Local Outlier Factor (LOF) detector to identify anomalous observations in the CATS time-series dataset.

LOF is an unsupervised anomaly detection algorithm that identifies observations whose local density differs substantially from the density of their neighbours. An observation may be considered anomalous when it lies in a region that is less dense than the regions surrounding it.

## Training and Evaluation Design

The detector will use the first 1,000,000 observations, which contain nominal data, as its reference training period. Its predictions will be evaluated against the remaining 4,000,000 observations.

The 17 telemetry signals will be used as model features. The ground-truth label `y`, anomaly category `category`, and event metadata will be reserved for evaluation and will not be used to train the detector.

## Evaluation Objectives

The detector will be assessed at two levels:

1. **Observation level:** Measure precision, recall, and false-alarm rate by comparing predicted anomalies with ground-truth labels.
2. **Event level:** Measure how many of the 200 known anomaly events are detected, the fraction of observations flagged within each event, and detection delay.

The results will eventually be compared with the rule-based, rolling statistical, and Isolation Forest detectors.

## Important Consideration

LOF identifies anomalies using local neighbourhood density. Unlike Isolation Forest, its computational cost can increase substantially with the size of the reference dataset. The training reference size and prediction strategy must therefore be chosen deliberately, with any sampling limitations documented when interpreting results.

# 2. Mathematical Foundation of Local Outlier Factor

## 2.1 Core Idea

Local Outlier Factor (LOF) identifies observations whose local density is substantially lower than the density of their neighbours.

Unlike a global threshold that assumes all normal observations follow the same range, LOF compares each observation with nearby observations. An observation may therefore be considered anomalous when it lies in a locally sparse region, even if its values are not globally extreme.

LOF uses four main concepts: the *k-distance*, the *reachability distance*, the *local reachability density*, and the *LOF score*.

## 2.2 k-Distance and Neighbourhood

For an observation \(x\), the \(k\)-distance is the distance from \(x\) to its \(k\)-th nearest neighbour.

Let \(N_k(x)\) denote the set of neighbours of \(x\), using the neighbourhood definition implemented by the algorithm.

The parameter \(k\) controls the neighbourhood size. A small \(k\) focuses on very local structure, while a larger \(k\) considers a broader region.

## 2.3 Reachability Distance

The reachability distance from observation \(x\) to a neighbour \(o\) is defined as

$$
\operatorname{reach\text{-}dist}_k(x,o)
=
\max\left\{
k\text{-distance}(o),\ d(x,o)
\right\}
$$

where:

- \(d(x,o)\) is the distance between \(x\) and \(o\).
- \(k\text{-distance}(o)\) is the distance from \(o\) to its \(k\)-th nearest neighbour.

This definition limits the effect of very small distances in dense regions.

## 2.4 Local Reachability Density

The local reachability density (LRD) of \(x\) is the inverse of its average reachability distance to its neighbours:

$$
\operatorname{LRD}_k(x)
=
\frac{1}{
\displaystyle
\frac{1}{|N_k(x)|}
\sum_{o\in N_k(x)}
\operatorname{reach\text{-}dist}_k(x,o)
}
$$

Here, \(|N_k(x)|\) is the number of neighbours in the neighbourhood.

A higher LRD indicates a denser local region. A lower LRD indicates a sparser local region.

## 2.5 Local Outlier Factor

The LOF score compares the local reachability density of an observation with the densities of its neighbours:

$$
\operatorname{LOF}_k(x)
=
\frac{1}{|N_k(x)|}
\sum_{o\in N_k(x)}
\frac{\operatorname{LRD}_k(o)}
{\operatorname{LRD}_k(x)}
$$

Interpretation:

- **LOF approximately 1:** The observation has a local density similar to that of its neighbours.
- **LOF greater than 1:** The observation is less locally dense than its neighbours and may be anomalous.
- **LOF substantially greater than 1:** The observation is considerably less locally dense than its neighbours and may be a stronger anomaly candidate.

These are interpretations, not definitive labels. The score alone does not establish whether an observation is a true anomaly.

## 2.6 Applying LOF to CATS

The model will use the 17 telemetry signals to calculate neighbourhood distances and identify observations with unusual local density.

Because the signals may have different scales, feature scaling must be considered before calculating distances. The scaling procedure must be fitted using training data only and then applied consistently to evaluation data.

For evaluation, we will use the known labels and event metadata only after generating predictions. They will not be used as model features.

## 2.7 Important Implementation Distinction

In scikit-learn, LOF can be used for outlier detection on the fitted dataset or in novelty-detection mode for new observations.

For this project, we need to evaluate previously unseen observations against a reference dataset of nominal training observations. Therefore, we will use **novelty-detection mode** (`novelty=True`).

The implementation and thresholding strategy will be defined in subsequent steps. We will not assume that an LOF score greater than 1 is automatically the final classification threshold; that threshold must be established explicitly.

# 3. Data Preparation

## Objective

Prepare the CATS telemetry data for Local Outlier Factor (LOF) while preserving the temporal separation between training and evaluation.

## 3.1 Feature Selection

The model will use the following 17 telemetry signals:

`aimp`, `amud`, `arnd`, `asin1`, `asin2`, `adbr`, `adfl`, `bed1`, `bed2`, `bfo1`, `bfo2`, `bso1`, `bso2`, `bso3`, `ced1`, `cfo1`, and `cso1`.

The ground-truth label `y`, anomaly category `category`, and event metadata will be excluded from the model features.

## 3.2 Temporal Split

The dataset will retain the established chronological split:

- **Training reference:** the first 1,000,000 observations, representing the nominal training period.
- **Evaluation:** the remaining 4,000,000 observations.

This separation ensures that evaluation observations are not used to fit the detector.

## 3.3 Feature Scaling

LOF relies on distances between observations. Features with larger numerical scales can disproportionately influence those distances.

We will therefore evaluate an appropriate feature-scaling approach. Any scaler must be fitted using training data only and then applied to the evaluation data using the same fitted parameters.

## 3.4 Computational Strategy

LOF can be expensive on large datasets because it requires neighbourhood searches. We will choose the reference training size and evaluation strategy deliberately rather than assume that fitting all 1,000,000 observations will be practical on a local laptop.

Any sampling used to make the computation feasible must be documented because it can affect the detector's results.

In [ ]:
from pathlib import Path

import pandas as pd

INPUT_ROOT = Path(
    "/kaggle/input/datasets/patrickfleith/"
    "controlled-anomalies-time-series-dataset"
)

DATA_PATH = INPUT_ROOT / "data.csv"
METADATA_PATH = INPUT_ROOT / "metadata.csv"

df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
event_metadata = pd.read_csv(METADATA_PATH)

signal_columns = [
    "aimp", "amud", "arnd", "asin1", "asin2",
    "adbr", "adfl", "bed1", "bed2", "bfo1",
    "bfo2", "bso1", "bso2", "bso3", "ced1",
    "cfo1", "cso1",
]

X_train = df.iloc[:1_000_000][signal_columns]
X_eval = df.iloc[1_000_000:][signal_columns]

y_eval = df.iloc[1_000_000:]["y"].astype("int8")
category_eval = df.iloc[1_000_000:]["category"]

print(f"Dataset shape: {df.shape}")
print(f"Training reference shape: {X_train.shape}")
print(f"Evaluation shape: {X_eval.shape}")
print(f"Event metadata shape: {event_metadata.shape}")
print(f"Training index type: {type(X_train.index).__name__}")
print(f"Evaluation index type: {type(X_eval.index).__name__}")
print(f"Training anomaly labels: {df.iloc[:1_000_000]['y'].sum()}")

Dataset shape: (5000000, 19)
Training reference shape: (1000000, 17)
Evaluation shape: (4000000, 17)
Event metadata shape: (200, 5)
Training index type: DatetimeIndex
Evaluation index type: DatetimeIndex
Training anomaly labels: 0.0


# 4. Feature Scaling and Reference Strategy

## 4.1 Feature Scaling

LOF compares observations using distances across the 17 telemetry signals. If these signals have different numerical scales, signals with larger ranges may disproportionately influence the neighbourhood structure.

We will use **standardization** to transform each feature using the mean and standard deviation calculated from the nominal training data.

The formula is:

$$
z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}
$$

**What does the formula mean?**

For each telemetry value, subtract the training-data average and divide the result by the training-data standard deviation.

| Symbol | Meaning |
|---|---|
| $x_{ij}$ | The original value of feature $j$ for observation $i$. |
| $\mu_j$ | The average value of feature $j$ in the training data. |
| $\sigma_j$ | How much the values of feature $j$ vary in the training data. |
| $z_{ij}$ | The standardized value, showing how far the original value is from the training average in standard-deviation units. |

**Example**

Suppose a telemetry signal has an original value of 80, a training average of 60, and a standard deviation of 10.

$$
z = \frac{80 - 60}{10} = 2
$$

The standardized value is 2, meaning the original value is two standard deviations above the training average.

The scaler will be fitted exclusively on the training data and then used to transform the evaluation data. This prevents information from the evaluation period from influencing the scaling parameters.

## 4.2 Reference Dataset Strategy

The LOF reference dataset will contain all 1,000,000 observations from the nominal training period. These observations contain 17 telemetry features, and the training period has been verified to contain no labelled anomalies.

Using the full training period preserves the available representation of normal telemetry behaviour rather than introducing a sampling decision that could affect the results.

LOF requires neighbourhood searches against the reference observations. Fitting and scoring at this scale may be computationally expensive, particularly when evaluating 4,000,000 observations. We will implement the full-reference approach and assess its computational feasibility on Kaggle rather than silently reducing the reference dataset.

The reference dataset will be used for model fitting only. The evaluation observations will not be included in the reference dataset.

## 4.3 Evaluation Separation

The evaluation observations will remain separate from the data used to fit the scaler and LOF reference model. Ground-truth labels and event metadata will be used only after predictions have been generated.

# 5. Feature Scaling Implementation

Before fitting the Local Outlier Factor (LOF) detector, the 17 telemetry features must be standardized so that differences in their numerical scales do not disproportionately influence distance calculations.

We use `StandardScaler` to standardize each feature based on its mean and standard deviation in the nominal training data.

The scaler is fitted exclusively on the 1,000,000 training observations. The fitted scaler is then applied to both the training and evaluation datasets.

The scaled training data will be used to fit the LOF reference model, while the scaled evaluation data will be used to score previously unseen observations.

After transformation, we verify the dimensions, feature means, feature standard deviations, and the presence of non-finite values. These checks help confirm that the scaling procedure completed successfully before model fitting.

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_eval_scaled = scaler.transform(X_eval)

print("Training reference shape:", X_train_scaled.shape)
print("Evaluation shape:", X_eval_scaled.shape)

print(
    "Training scaled feature means (first 5):",
    X_train_scaled.mean(axis=0)[:5].round(4)
)

print(
    "Training scaled feature standard deviations (first 5):",
    X_train_scaled.std(axis=0)[:5].round(4)
)

print("All training values finite:", __import__("numpy").isfinite(X_train_scaled).all())
print("All evaluation values finite:", __import__("numpy").isfinite(X_eval_scaled).all())

KeyboardInterrupt: 

# 6. LOF Model Configuration

The Local Outlier Factor (LOF) detector will be configured for novelty detection. This allows the model to learn the local-density structure of the nominal training data and score previously unseen observations.

The `n_neighbors` parameter specifies the number of neighboring observations used to estimate local density. This choice affects the scale of anomalies the detector can identify.

The `novelty=True` setting is required for our evaluation approach because the model will score observations that were not used during fitting.

The detector will use the full 1,000,000-observation training reference, as previously decided. Before fitting, we must account for the computational cost of building and querying neighborhoods at this scale.

The evaluation labels and event metadata will remain excluded from model fitting and configuration.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=20,
    algorithm="auto",
    contamination="auto",
    novelty=True,
    n_jobs=-1,
)

print("LOF configuration")
print("Number of neighbors:", lof.n_neighbors)
print("Search algorithm:", lof.algorithm)
print("Contamination:", lof.contamination)
print("Novelty detection:", lof.novelty)
print("Parallel jobs:", lof.n_jobs)
print("Training reference observations:", X_train_scaled.shape[0])
print("Features:", X_train_scaled.shape[1])
print("Evaluation observations:", X_eval_scaled.shape[0])

# 7. LOF Model Fitting

The LOF detector will be fitted using the standardized nominal training reference, `X_train_scaled`. The model will estimate the local-density structure of the training observations using the configured neighborhood size.

The evaluation observations will not be included during fitting. They will be scored separately after the reference model has been fitted.

Because the reference contains 1,000,000 observations and the evaluation dataset contains 4,000,000 observations, computational feasibility is a material consideration. We will assess the fitting and scoring requirements before launching the full computation.

The training reference size will not be reduced without an explicit decision and documentation of the methodological consequences.

In [ ]:
import inspect
from sklearn.neighbors import LocalOutlierFactor

print("fit_predict available:", hasattr(LocalOutlierFactor, "fit_predict"))
print("score_samples available:", hasattr(LocalOutlierFactor, "score_samples"))
print("decision_function available:", hasattr(LocalOutlierFactor, "decision_function"))
print("Novelty mode enabled:", lof.novelty)
print("Configured neighbors:", lof.n_neighbors)

In [ ]:
import time
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=20,
    algorithm="kd_tree",
    contamination="auto",
    novelty=True,
    n_jobs=-1,
)

start_time = time.perf_counter()

lof.fit(X_train_scaled)

fit_time = time.perf_counter() - start_time

print("LOF fitting completed successfully.")
print(f"Reference observations: {X_train_scaled.shape[0]:,}")
print(f"Features: {X_train_scaled.shape[1]}")
print(f"Number of neighbors: {lof.n_neighbors}")
print(f"Neighbor-search algorithm: {lof._fit_method}")
print(f"Fitting time: {fit_time:.2f} seconds")
print(f"Fitting time: {fit_time / 60:.2f} minutes")

# 8. Anomaly Scoring

The fitted LOF model will score the 4,000,000 evaluation observations using the `score_samples()` method available in novelty-detection mode.

Scikit-learn returns the negative outlier factor for each observation through this method. More negative scores indicate observations that are more anomalous relative to the learned reference distribution.

For easier interpretation, we will reverse the sign of the scores:

$$
s(x) = -\operatorname{score\_samples}(x)
$$

Higher values of $s(x)$ will therefore indicate greater anomaly suspicion.

Evaluation labels and event metadata will not be used to generate these scores. They will be reserved for subsequent evaluation.

Because the evaluation dataset contains 4,000,000 observations, scoring will be performed in batches to limit temporary memory requirements. We will record the scoring time and verify the resulting score distribution before selecting an anomaly threshold.

In [ ]:
import time
import numpy as np

batch_size = 100_000
n_eval = X_eval_scaled.shape[0]

lof_scores = np.empty(n_eval, dtype=np.float32)

start_time = time.perf_counter()

for start in range(0, n_eval, batch_size):
    end = min(start + batch_size, n_eval)

    batch_scores = lof.score_samples(X_eval_scaled[start:end])
    lof_scores[start:end] = -batch_scores

    print(f"Scored {end:,} / {n_eval:,} evaluation observations")

scoring_time = time.perf_counter() - start_time

print("\nAnomaly scoring completed.")
print(f"Evaluation observations scored: {len(lof_scores):,}")
print(f"Scoring time: {scoring_time:.2f} seconds")
print(f"Scoring time: {scoring_time / 60:.2f} minutes")
print(f"All scores finite: {np.isfinite(lof_scores).all()}")
print(f"Minimum anomaly score: {lof_scores.min():.6f}")
print(f"Maximum anomaly score: {lof_scores.max():.6f}")
print(f"Mean anomaly score: {lof_scores.mean():.6f}")
print(f"Median anomaly score: {np.median(lof_scores):.6f}")